# Part 2: Entities and graphs

A Pantry Lane customer has a chocolate wrapper with batch code **EC26-0811** and wants to know which farms grew its cacao. A single passage can mention a batch or a farm, but a graph makes the connection explicit:

**Batch → BeanLot ← Farm**

**You'll:**
- Read the **ontology**, which defines the entity types and relationships in the graph.
- Compare a source document with a stored graph **assertion** (a subject, predicate, and object).
- Write a **GQL** query that follows the batch to its farms, and trace the result back to its evidence.
- Ask the same question through a graph-backed knowledge base and through the standard knowledge base from Part 1.

Run Part 1 first, because this notebook reuses its knowledge sources and standard knowledge base. You can use the **Cocoarynth graph explorer** app alongside this notebook to see the ontology and graph visually. The lab instructions explain how to start it.

## Step 1: Load your settings

Load the same settings as in Part 1.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import json
import importlib.metadata
from urllib.parse import urlsplit
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
import azure.search.documents as search_documents
from dotenv import load_dotenv
import os
from pathlib import Path

from azure.core.credentials import AzureKeyCredential

REPO_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / ".env.sample").is_file() and (directory / "src" / "notebooks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within the LAB673 repository.")

SDK_VERSION = "12.1.0a20261027002"
installed_version = importlib.metadata.version("azure-search-documents")
if installed_version != SDK_VERSION or search_documents.__version__ != SDK_VERSION:
    raise RuntimeError(
        f"Use azure-search-documents=={SDK_VERSION}; found installed {installed_version}, "
        f"imported {search_documents.__version__}. Restart the kernel after changing packages."
    )
load_dotenv(REPO_ROOT / ".env", override=False)
for setting in ('AZURE_SEARCH_SERVICE_ENDPOINT', 'AZURE_SEARCH_ADMIN_KEY', 'AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_KEY', 'AZURE_OPENAI_CHATGPT_DEPLOYMENT', 'AZURE_OPENAI_CHATGPT_MODEL_NAME'):
    value = os.environ.get(setting, "").strip()
    if not value or "your-" in value:
        raise RuntimeError(f"Set {setting} in your environment or the root .env file.")

def check_response(response) -> None:
    status = response.http_response.status_code
    if status == 206:
        raise RuntimeError("Search returned HTTP 206: these results are incomplete.")
    if status < 400:
        return
    try:
        detail = json.loads(response.http_response.text())["error"]["message"]
    except (json.JSONDecodeError, KeyError, TypeError):
        detail = "The service did not return a JSON error message."
    if not isinstance(detail, str):
        detail = "The service did not return a text error message."
    for secret in (
        response.http_request.headers.get("api-key", ""),
        os.environ.get("AZURE_OPENAI_KEY", ""),
    ):
        if secret:
            detail = detail.replace(secret, "[REDACTED]")
    guidance = {
        402: " Ask the service owner about the Standard Knowledge Retrieval prerequisite.",
        412: " Another request changed this resource. Rerun the cell to read its current state.",
        429: " The service is busy or its allowance is exhausted. Retry later.",
    }.get(status, "")
    error_type = ResourceNotFoundError if status == 404 else HttpResponseError
    raise error_type(f"Search returned HTTP {status}: {detail}{guidance}") from None


CLIENT_OPTIONS = {
    "retry_total": 0,
    "permit_redirects": False,
    "connection_timeout": 15,
    "read_timeout": 120,
    "raw_response_hook": check_response,
}

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"].rstrip("/")
origin = urlsplit(AZURE_SEARCH_SERVICE_ENDPOINT)
if (origin.scheme != "https" or not origin.hostname or origin.username or origin.password
        or origin.port not in (None, 443) or origin.path not in ("", "/")
        or origin.query or origin.fragment):
    raise ValueError("The Search endpoint must be an HTTPS service origin.")
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print(f"Environment variables loaded; Search SDK: {SDK_VERSION}")

## Step 2: Connect to the graph indexes

The graph is stored in three indexes:

- **cocoarynth-ontology**: The entity types and relationships the graph can contain.
- **cocoarynth-graph**: The assertions themselves, one row per subject-predicate-object fact.
- **cocoarynth-graph-sources**: The source passages and records that each assertion came from.

A **graph knowledge source** (`cocoarynth-graph-ks`) connects the graph index and its ontology to a knowledge base.

In [ ]:
import json
import textwrap
import time

from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient


GRAPH_INDEX = "cocoarynth-graph"
ONTOLOGY_INDEX = "cocoarynth-ontology"
EVIDENCE_INDEX = "cocoarynth-graph-sources"
GRAPH_KNOWLEDGE_SOURCE = f"{GRAPH_INDEX}-ks"

index_client = SearchIndexClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, search_credential, **CLIENT_OPTIONS
)
graph_client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, GRAPH_INDEX, search_credential, **CLIENT_OPTIONS)
ontology_client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, ONTOLOGY_INDEX, search_credential, **CLIENT_OPTIONS)
evidence_client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, EVIDENCE_INDEX, search_credential, **CLIENT_OPTIONS)

for name in (GRAPH_INDEX, ONTOLOGY_INDEX, EVIDENCE_INDEX):
    print(f"{name}: {index_client.get_index(name).name} found")
print(f"{GRAPH_KNOWLEDGE_SOURCE}: {index_client.get_knowledge_source(GRAPH_KNOWLEDGE_SOURCE).name} found")

## Step 3: Read the ontology

An **ontology** describes the kinds of things in the graph and how they can relate. For example, a `Batch` can be `madeFrom` a `BeanLot`, and a `Farm` can have `contributedTo` a `BeanLot`. A relationship's **domain** is its subject type and its **range** is its object type.

The ontology is a schema. It describes types such as `Farm`, not individual farms, which you'll find in the graph itself.

> **Tip:** Open the **Ontology** tab in the graph explorer app to see all the entity types and relationships as a diagram.

In [ ]:
entity_types = ontology_client.search(
    search_text="*",
    filter="subject_type eq 'NodeType' and predicate eq 'rdfs:comment'",
    select=["subject", "object"],
    top=1000,
)
print("Entity types:")
for row in sorted(entity_types, key=lambda row: row["subject"]):
    print(f"  {row['subject']}: {row['object']}")

relationships = {}
for row in ontology_client.search(
    search_text="*",
    filter="subject_type eq 'EdgeType' and search.in(subject, 'madeFrom,contributedTo,concernsBatch')",
    select=["subject", "predicate", "object"],
    top=100,
):
    relationships.setdefault(row["subject"], {})[row["predicate"]] = row["object"]

print("\nRelationships used in this notebook:")
for name, parts in sorted(relationships.items()):
    print(f"  {parts['rdfs:domain']} --{name}--> {parts['rdfs:range']}: {parts['rdfs:comment']}")

## Step 4: Compare a source document with a graph assertion

Open [batch record EC26-0811](../../../data/cocoarynth/documents/cocoarynth-quality/batch-record-ec26-0811.pdf). Find the batch code, bean lot, and input quantity.

The cell below shows the matching passage from the quality index, and then the graph assertion that says which bean lot the batch was made from. Each assertion records the source it came from. In this graph, the `madeFrom` assertion comes from a structured Cocoarynth record, not from the PDF itself.

In [ ]:
quality_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, "cocoarynth-quality", search_credential, **CLIENT_OPTIONS
)
print("Passages that mention EC26-0811 in the quality index:")
for passage in quality_client.search(
    search_text='"EC26-0811"', search_fields=["chunk"], select=["title", "blob_path", "chunk"], top=3
):
    print(f"- {passage['title']} ({passage['blob_path']})")
    print(textwrap.indent(textwrap.shorten(passage["chunk"], 300), "    "))

fact = next(iter(graph_client.search(
    search_text="*",
    filter="subject_id eq 'urn:cocoarynth:Batch:EC26-0811' and predicate eq 'madeFrom'",
    top=1,
)))
print(f"\nGraph assertion: {fact['subject']} --{fact['predicate']}--> {fact['object']}")

source = evidence_client.get_document(fact["chunk_id"])
print(f"\nSource of this assertion: {source.get('title')} ({source.get('blob_path')})")
print(f"Pointer: {source.get('source_pointer')}")
print(textwrap.indent(textwrap.shorten(source["chunk"], 400), "    "))

## Step 5: Break the assertion into its parts

An assertion is a **triple**: a subject, a predicate, and an object. This one connects two entities. Other assertions connect an entity to a literal value, such as a quantity or a date.

Each entity has a display name and a stable ID. The ID is what identifies the same entity across different records. The graph index's **graph profile** tells the service which index fields hold each part of the triple.

In [ ]:
for field in ("subject", "subject_type", "subject_id", "predicate", "object", "object_type", "object_id", "chunk_id"):
    print(f"{field:<14} {fact.get(field)}")

graph_profile = index_client.get_index(GRAPH_INDEX).as_dict()["graphProfile"]
print("\nGraph profile field mapping:")
for role, field_name in graph_profile["fields"].items():
    print(f"  {role:<22} -> {field_name}")

## Step 6: Run a one-hop GQL query

GQL (Graph Query Language) describes a pattern to match in the graph:

- `MATCH` describes the path. Parentheses are nodes, and arrows are relationships.
- `WHERE` picks a specific entity by its stable ID.
- `RETURN` chooses the columns.
- `LIMIT` caps the number of rows.

`SearchClient.graph()` runs the GQL through the SDK's native graph operation and returns typed rows, plans, statistics, and provenance. No raw HTTP fallback is needed.

> **Tip:** In the graph explorer app, open the **Graph** tab and search for `EC26-0811` to see this entity and its neighbors.

In [ ]:
from azure.search.documents.models import GraphQueryProvenanceParameters


def run_graph_query(gql, plan_only=False):
    return graph_client.graph(
        gql=gql,
        execution_mode="planOnly" if plan_only else "execute",
        explain=["physicalPlan"],
        include_statistics=True,
        provenance=GraphQueryProvenanceParameters(kind="ids", max_evidence_per_row=20),
    )


GQL_ONE_HOP = """
MATCH (b:`Batch`)-[:`madeFrom`]->(l:`BeanLot`)
WHERE b.id = 'urn:cocoarynth:Batch:EC26-0811'
RETURN b.name AS batch, l.name AS lot, l.id AS lot_id
LIMIT 10
"""
for row in run_graph_query(GQL_ONE_HOP).value:
    print(f"{row['batch']} --madeFrom--> {row['lot']} ({row['lot_id']})")

## Step 7: Join two relationships through the same bean lot

Now extend the pattern with farms that contributed to the bean lot. Both relationships share the variable `l`, so they join on the **same** bean lot:

`(b)-[:madeFrom]->(l)<-[:contributedTo]-(f)` reads as **Batch → BeanLot ← Farm**.

In [ ]:
GQL_FARMS = """
MATCH (b:`Batch`)-[:`madeFrom`]->(l:`BeanLot`)<-[:`contributedTo`]-(f:`Farm`)
WHERE b.id = 'urn:cocoarynth:Batch:EC26-0811'
RETURN b.id AS batch_id, b.name AS batch,
       l.id AS lot_id, l.name AS lot,
       f.id AS farm_id, f.name AS farm
LIMIT 10
"""
print(GQL_FARMS)

## Step 8: Look at the query plan

With `executionMode` set to `planOnly`, the service plans the query without running it, so no rows come back. The plan shows how the service will look up each part of the pattern.

In [ ]:
planned = run_graph_query(GQL_FARMS, plan_only=True)
print(f"Rows returned: {len(planned.value)}")
print(json.dumps(planned.explain.as_dict() if planned.explain is not None else {}, indent=2))
print(json.dumps(planned.statistics.as_dict() if planned.statistics is not None else {}, indent=2))

## Step 9: Run the query

Run the query to find the farms behind batch EC26-0811. The statistics show how many searches the service ran and how long they took.

In [ ]:
start = time.perf_counter()
joined = run_graph_query(GQL_FARMS)
graph_query_ms = (time.perf_counter() - start) * 1000
paths = joined.value

for row in paths:
    print(f"{row['batch']} --madeFrom--> {row['lot']} <--contributedTo-- {row['farm']}")
print(json.dumps(joined.statistics.as_dict() if joined.statistics is not None else {}, indent=2))

## Step 10: Follow each path back to its evidence

Each returned row includes **provenance**: the IDs of the assertions that support it and the source chunks they came from. Follow those IDs to read the actual evidence.

The evidence count includes a relation: `exact` or `lowerBound`. Read that relation rather than assuming the bounded evidence collection contains every assertion, or every record that might mention the same fact.

In [ ]:
for row in paths:
    provenance = row.provenance
    if provenance is None or not provenance.evidence:
        raise RuntimeError("The graph query did not return the requested assertion evidence.")
    print(f"{row['farm']} (evidence count: {provenance.evidence_count.as_dict()})")
    for item in provenance.evidence:
        assertion = graph_client.get_document(item.edge_id)
        source = evidence_client.get_document(item.chunk_id)
        print(f"  {assertion['subject']} --{assertion['predicate']}--> {assertion['object']}")
        print(f"    source: {source.get('blob_path')} {source.get('source_pointer') or ''}")
        print(textwrap.indent(textwrap.shorten(source["chunk"], 200), "      "))

## Step 11: Ask a graph-backed knowledge base

You wrote the GQL above by hand. Now create a knowledge base over the graph knowledge source and ask the question in natural language. The knowledge base can use a **graph search** tool that follows relationships in the graph, as well as ordinary text search.

The retrieval instructions encourage the model to use graph search, but they don't force it. The next cell checks the activity to confirm that graph search actually ran.

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
    SearchIndexKnowledgeSourceParams,
)

GRAPH_KB_NAME = "cocoarynth-kb-graph"
graph_knowledge_base = KnowledgeBase(
    name=GRAPH_KB_NAME,
    description="Relationship reasoning over the Cocoarynth graph knowledge source",
    knowledge_sources=[KnowledgeSourceReference(name=GRAPH_KNOWLEDGE_SOURCE)],
    models=[
        KnowledgeBaseMicrosoftRetriever1Model(),
        KnowledgeBaseAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=AZURE_OPENAI_ENDPOINT,
                deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
                model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
                api_key=AZURE_OPENAI_KEY,
            )
        )
    ],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions=(
        "For relationship questions use the graph_search tool and the supplied ontology. "
        "For batch traceability follow Batch-madeFrom->BeanLot<-contributedTo-Farm. "
        "Use stable IDs when provided."
    ),
    answer_instructions="Answer only from returned evidence, cite reference IDs, and say when evidence is missing.",
)
index_client.create_or_update_knowledge_base(graph_knowledge_base)
print(f"Knowledge base '{GRAPH_KB_NAME}' created or updated.\n")

RELATIONSHIP_QUESTION = "Which farms contributed cacao to the bean lot used in batch EC26-0811? Give the farm names."


def relationship_request(knowledge_source_names):
    return KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=RELATIONSHIP_QUESTION)])
        ],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
        include_activity=True,
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=name,
                include_references=True,
                include_reference_source_data=True,
                always_query_source=True,
            )
            for name in knowledge_source_names
        ],
    )


graph_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=GRAPH_KB_NAME,
    credential=search_credential,
    **CLIENT_OPTIONS,
)
start = time.perf_counter()
graph_result = graph_kb_client.retrieve(relationship_request([GRAPH_KNOWLEDGE_SOURCE]))
graph_kb_ms = (time.perf_counter() - start) * 1000
display(Markdown(graph_result.response[0].content[0].text))

print("Activity:")
for activity in graph_result.activity:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<32} results: {step.get('count', '-')}")

print("\nReferences:")
for reference in graph_result.references:
    source_data = reference.source_data or {}
    print(f"  [{reference.id}] {source_data.get('title', reference.type)}")
    if "chunk" in source_data:
        print(textwrap.indent(textwrap.shorten(source_data["chunk"], 200), "      "))

### Check that graph search actually ran

Look for a `graphSearch` step in the activity that succeeded and returned results. If the service returns the GQL it generated, it's shown here. Otherwise, remember that the GQL earlier in this notebook was written by hand, not by the model.

In [ ]:
graph_steps = [
    step.as_dict() for step in graph_result.activity or []
    if step.as_dict().get("type") == "graphSearch"
]
if not graph_steps:
    print("No graph search ran for this question. The answer came from text search only.")
for step in graph_steps:
    arguments = step.get("graphSearchArguments") or {}
    print(f"graphSearch: outcome={arguments.get('outcome')}, results={step.get('count')}, {step.get('elapsedMs')} ms")
    if arguments.get("gql"):
        print(arguments["gql"])
    else:
        print("The service didn't return the generated GQL for this step.")

## Step 12: Compare with ordinary document retrieval

Ask the same question using the standard knowledge base from Part 1, which searches the original documents.

Ordinary retrieval may answer correctly too. What the graph adds is an explicit, inspectable path between entities, with provenance for each link. It doesn't guarantee a better answer or a faster one.

In [ ]:
STANDARD_KB_NAME = "cocoarynth-kb-standard"
document_knowledge_sources = [
    f"cocoarynth-{domain}-source" for domain in ("origins", "quality", "logistics-retail", "policies")
]

standard_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=STANDARD_KB_NAME,
    credential=search_credential,
    **CLIENT_OPTIONS,
)
start = time.perf_counter()
document_result = standard_client.retrieve(relationship_request(document_knowledge_sources))
document_kb_ms = (time.perf_counter() - start) * 1000
display(Markdown(document_result.response[0].content[0].text))

print("Activity:")
for activity in document_result.activity:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<32} results: {step.get('count', '-')}")

print("\nReferences:")
for reference in document_result.references:
    source_data = reference.source_data or {}
    print(f"  [{reference.id}] {source_data.get('title', reference.type)}")
    if "chunk" in source_data:
        print(textwrap.indent(textwrap.shorten(source_data["chunk"], 200), "      "))

print("\nResponse times (one request each):")
print(f"  Handwritten GQL query:       {graph_query_ms:>6.0f} ms  (graph traversal only, no model)")
print(f"  Graph knowledge base:        {graph_kb_ms:>6.0f} ms  (model planning, graph search, answer)")
print(f"  Standard knowledge base:     {document_kb_ms:>6.0f} ms  (model planning, document search, answer)")

## Summary

A graph makes relationship questions inspectable: typed entities, stable IDs, explicit joins, and provenance for each link. Natural-language graph retrieval can take longer than a handwritten query because the model plans the search. Stored relationships can also be incomplete, so check the original records before relying on them.

For a full customer reply, also check [the data-sharing policy](../../../data/cocoarynth/documents/cocoarynth-policies/data-sharing-policy.pdf). Knowing which farm grew the cacao doesn't mean you can share personal details about the people there.

**Try it yourself:** Trace the July complaint to its depot stop and storage site. Use the ontology tab in the graph explorer app to find the relationships to follow.

Continue to [Part 3: Topics](03-topics.ipynb).